<a href="https://colab.research.google.com/github/amit-sahu-a11y/ML_projects_for_practice/blob/main/Support_Ticket_Classification_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support Ticket Classification
This notebook uses **TF-IDF + Multinomial Naive Bayes** to classify support tickets into:
- Billing
- Technical
- HR
- General

Bonus:
- Confidence score
- Human review (<60%)
- Priority tagging
- Reflection


In [1]:
import pandas as pd
import re,string
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

df=pd.read_csv("tickets_large.csv")
df["text"]=df["subject"]+" "+df["body"]
df.head()


,subject,body,category,text
0,Office address,Share office location. Ticket reference 1024.,General,Office address Share office location. Ticket r...
1,Office address,Share office location. Ticket reference 1015.,General,Office address Share office location. Ticket r...
2,Invoice missing,Please send my latest invoice. Ticket referenc...,Billing,Invoice missing Please send my latest invoice....
3,Interview schedule,When is my interview? Ticket reference 1040.,HR,Interview schedule When is my interview? Ticke...
4,Server error,Receiving 500 internal server error. Ticket re...,Technical,Server error Receiving 500 internal server err...


In [2]:
def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+","",text)
    text=text.translate(str.maketrans("","",string.punctuation))
    text=re.sub(r"\d+","",text)
    text=re.sub(r"\s+"," ",text).strip()
    return text

df["text"]=df["text"].apply(clean_text)

X_train,X_test,y_train,y_test=train_test_split(
    df["text"],df["category"],test_size=0.2,random_state=42,stratify=df["category"])

model=Pipeline([
    ("tfidf",TfidfVectorizer(stop_words="english")),
    ("clf",MultinomialNB())
])

model.fit(X_train,y_train)
pred=model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
print("Confusion Matrix")
print(confusion_matrix(y_test,pred))


Accuracy: 1.0
              precision    recall  f1-score   support

     Billing       1.00      1.00      1.00        10
     General       1.00      1.00      1.00        10
          HR       1.00      1.00      1.00        10
   Technical       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix
[[10  0  0  0]
 [ 0 10  0  0]
 [ 0  0 10  0]
 [ 0  0  0 10]]


In [3]:
urgent_keywords={"urgent","down","critical","immediately","not working","failed","error"}

def predict_ticket(ticket):
    cleaned=clean_text(ticket)
    pred=model.predict([cleaned])[0]
    probs=model.predict_proba([cleaned])[0]
    classes=model.named_steps["clf"].classes_
    confidence=max(probs)
    priority="Urgent" if any(k in cleaned for k in urgent_keywords) else "Normal"
    review="Needs Human Review" if confidence<0.60 else "Auto Assigned"
    print(f"Ticket: {ticket}")
    print(f"Category: {pred}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Priority: {priority}")
    print(f"Status: {review}")
    print("-"*40)

samples=[
"My payment failed and I was charged twice.",
"The website is down and shows server error.",
"I need my salary slip for July.",
"What are your office timings?",
"Password reset is not working urgently."
]

for s in samples:
    predict_ticket(s)


Ticket: My payment failed and I was charged twice.
Category: Billing
Confidence: 81.24%
Priority: Urgent
Status: Auto Assigned
----------------------------------------
Ticket: The website is down and shows server error.
Category: Technical
Confidence: 80.73%
Priority: Urgent
Status: Auto Assigned
----------------------------------------
Ticket: I need my salary slip for July.
Category: HR
Confidence: 70.00%
Priority: Normal
Status: Auto Assigned
----------------------------------------
Ticket: What are your office timings?
Category: General
Confidence: 70.06%
Priority: Normal
Status: Auto Assigned
----------------------------------------
Ticket: Password reset is not working urgently.
Category: Technical
Confidence: 75.36%
Priority: Urgent
Status: Auto Assigned
----------------------------------------
